<a href="https://colab.research.google.com/github/chigur-DA/Executive-B2B-Sales-Pipeline-Revenue-Analysis/blob/main/crm_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.cloud import bigquery
import pandas as pd
from google.colab import auth

auth.authenticate_user()

client = bigquery.Client(project='b2b-sales-pipeline-crm')
query = """
SELECT
    opportunity_id,
    sp.sales_agent,
    sp.product,
    sp.account,
    sp.deal_stage,
    sp.engage_date,
    sp.close_date,
    sp.close_value,
    p.series as product_series,
    ac.office_location as company_office_location,
    ac.sector AS sector,
    st.regional_office as sales_agent_regional_office
FROM `sales_data.sales_pipeline` as sp
LEFT JOIN `sales_data.accounts` as ac ON sp.account=ac.account
LEFT JOIN `sales_data.products` as p ON sp.product=p.product
LEFT JOIN `sales_data.sales_teams` as st ON sp.sales_agent=st.sales_agent
"""


df = client.query(query).to_dataframe()
df.head()

,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value,product_series,company_office_location,sector,sales_agent_regional_office
0,LAYVBSH4,Anna Snelling,GTX Basic,None,Engaging,2017-07-19,NaT,<NA>,GTX,None,None,Central
1,055Z2OAS,Anna Snelling,MG Special,Dontechi,Engaging,2017-07-19,NaT,<NA>,MG,United States,software,Central
2,Q9VC6GV0,Anna Snelling,MG Special,Konmatfix,Engaging,2017-07-19,NaT,<NA>,MG,United States,marketing,Central
3,ZGFB0IF1,Anna Snelling,GTX Plus Basic,None,Engaging,2017-07-20,NaT,<NA>,GTX,None,None,Central
4,LD4F6QTP,Anna Snelling,GTX Plus Pro,Groovestreet,Engaging,2017-07-20,NaT,<NA>,GTX,United States,retail,Central


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8800 entries, 0 to 8799
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   opportunity_id               8800 non-null   object 
 1   sales_agent                  8800 non-null   object 
 2   product                      8800 non-null   object 
 3   account                      7375 non-null   object 
 4   deal_stage                   8800 non-null   object 
 5   engage_date                  8300 non-null   dbdate 
 6   close_date                   6711 non-null   dbdate 
 7   close_value                  6711 non-null   Int64  
 8   product_series               7320 non-null   object 
 9   company_office_location      8800 non-null   object 
 10  sector                       8800 non-null   object 
 11  sales_agent_regional_office  8800 non-null   object 
 12  sales_cycle_days             6711 non-null   float64
dtypes: Int64(1), dbdat

In [ ]:
df['engage_date'] = pd.to_datetime(df['engage_date'])
df['close_date'] = pd.to_datetime(df['close_date'])

In [ ]:
# тривалість циклу продажів у днях
df['sales_cycle_days'] = (df['close_date'] - df['engage_date']).dt.days
df[df['deal_stage'] == 'Won'].head()

,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value,product_series,company_office_location,sector,sales_agent_regional_office,sales_cycle_days
240,185PGG0J,Anna Snelling,MG Special,Treequote,Won,2017-07-14,2017-07-17,44,MG,United States,telecommunications,Central,3.0
241,233HYDSP,Anna Snelling,MG Special,Gogozoom,Won,2017-08-29,2017-12-03,44,MG,United States,telecommunications,Central,96.0
242,VYJ20GOM,Anna Snelling,MG Special,Cancity,Won,2017-09-11,2017-11-17,44,MG,United States,retail,Central,67.0
243,Q1ZKBGV5,Anna Snelling,MG Special,Globex Corporation,Won,2017-03-03,2017-05-23,46,MG,Norway,technolgy,Central,81.0
244,IF8PILJR,Anna Snelling,MG Special,Globex Corporation,Won,2017-07-27,2017-07-29,46,MG,Norway,technolgy,Central,2.0


In [ ]:
df[df['deal_stage'] == 'Won'].describe()

,engage_date,close_date,close_value,sales_cycle_days
count,4238,4238,4238.0,4238.000000
mean,2017-06-09 12:50:58.046248192,2017-07-31 07:37:00.481359104,2360.909391,51.781973
min,2016-10-20 00:00:00,2017-03-01 00:00:00,38.0,1.000000
25%,2017-03-25 00:00:00,2017-05-18 00:00:00,518.0,9.000000
50%,2017-06-12 00:00:00,2017-08-02 00:00:00,1117.0,57.000000
75%,2017-09-01 00:00:00,2017-10-16 00:00:00,4429.5,88.000000
max,2017-12-27 00:00:00,2017-12-31 00:00:00,30288.0,138.000000
std,NaN,NaN,2544.484463,41.171603


In [ ]:
df['sector'] = df['sector'].fillna('Unknown')
df['company_office_location'] = df['company_office_location'].fillna('Unknown')

In [ ]:
df['is_won'] = (df['deal_stage'] == 'Won').astype(int)

In [ ]:
df.to_csv('crm_sales_data_clean.csv')


https://public.tableau.com/app/profile/.17026898/viz/ExecutiveSalesPipelineDashboard/Dashboard1
